In [ ]:
import os
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.llms import LlamaCpp 
from langchain.chains import RetrievalQA
import uuid


def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [ ]:
local_file_path = r"C:\Users\faisa\Downloads\34e0a671.txt"
print(f"Processing local file: {local_file_path}")

In [ ]:
# 3. LOAD THE DOCUMENT USING TEXTLOADER WITH THE LOCAL PATH 
print(f"Loading '{local_file_path}' using LangChain's TextLoader...")
if not os.path.exists(local_file_path):
    print(f"ERROR: File not found at '{local_file_path}'. Please double-check the path.")
    exit() 

try:
    loader = TextLoader(local_file_path, encoding='utf-8') 
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} LangChain Document object(s).")
    if not documents:
        print("Warning: No documents loaded by TextLoader. The file might be empty.")
        exit() 
except Exception as e:
    print(f"An error occurred during document loading: {e}")
    exit() 

In [ ]:
# 4. SPLIT DOCUMENTS 
print("Splitting document(s) into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len
)
texts = text_splitter.split_documents(documents)
print(f"Split into {len(texts)} chunks.")
if not texts:
    print("Warning: No text chunks created. The document might be too short or an issue occurred.")

In [ ]:
# 5. INITIALIZing LLM 
print("Initializing Llama-2-7B-Chat LLM via LlamaCpp...")
model_path_llamacpp = "Done" 
llm = LlamaCpp(
    model_path=model_path_llamacpp,
    n_gpu_layers=0, n_batch=512, n_ctx=4096,
    f16_kv=True, temperature=0.1, max_tokens=512, verbose=True)
print("Llama-2-7B-Chat LLM initialized.")

In [ ]:

if 'texts' not in locals() or not texts:
    print("Error: 'texts' (document chunks) is not defined or is empty. Cannot proceed.")
elif 'llm' not in locals() or llm is None:
    print("Error: 'llm' (LlamaCpp model) is not defined or initialized. Cannot proceed.")
else:
    # Step 6 Embeddings, Chroma Vector Store, Retriever, and QA Chain 
    print("Step 6: Initializing Embeddings, Vector Store, Retriever, and QA Chain")

    print("Initializing HuggingFace embeddings...")
    embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
    print(f"Embeddings initialized with model: {embedding_model_name}")

    unique_collection_name = f"rag_collection_{uuid.uuid4().hex}"
    print(f"Creating Chroma vector store (docsearch) with unique collection name: {unique_collection_name}...")
    docsearch = Chroma.from_documents(
        documents=texts,
        embedding=embeddings,
        collection_name=unique_collection_name,
        persist_directory=None 
    )
    print(f"Chroma vector store (docsearch) created with collection: {unique_collection_name}")

    print("Creating retriever from vector store...")
    retriever = docsearch.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 chunks
    print("Retriever is ready.")

    print("Creating RetrievalQA chain...")
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=False # Set to True if you want to see the source documents
    )
    print("RetrievalQA chain (qa) ready.")

    # Step 7 User Asks Questions 
    print("\nStep 7: Querying the RAG system...")
    query = "Is physiotherapy covered?"
    print(f"\nQuestion: {query}")

    try:
        result = qa.invoke(query)
        if isinstance(result, dict) and "result" in result:
            print("\nAnswer from LLM:")
            print(result["result"])
        elif isinstance(result, str): # Some chains might return string directly
            print("\nAnswer from LLM:")
            print(result)
        else: # If the result is neither a dict with "result" nor a string
            print("\nReceived result (structure might be unexpected):")
            print(result)
    except Exception as e:
        # If direct string invocation fails, try with a dictionary input
        print(f"Error during qa.invoke(query) with direct string: {e}")
        print("Attempting qa.invoke({'query': query}) instead...")
        try:
            result_dict = qa.invoke({"query": query})
            print("\nAnswer from LLM (using dict input):")
            if "result" in result_dict:
                print(result_dict["result"])
            else:
                print("Result from dict input (key 'result' not found, full output):")
                print(result_dict)
        except Exception as e2:
            print(f"Error also during qa.invoke({{'query': query}}): {e2}")

print("\n--- RAG Querying Finished ---")